In [ ]:
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB with persistence
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(
        allow_reset=True,
        anonymized_telemetry=False
    )
)


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings

class DocumentProcessor:
    def __init__(self):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " "]
        )
        self.embeddings = OpenAIEmbeddings()

    def process_documents(self, documents):
        chunks = self.text_splitter.split_documents(documents)
        return self.embeddings.embed_documents([chunk.page_content for chunk in chunks])

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, WebsiteSearchTool
from langchain_openai import ChatOpenAI

# Initialize the language model
llm = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0.1)

# Research Agent
research_agent = Agent(
    role="Research Specialist",
    goal="Retrieve and analyze relevant documents from the knowledge base",
    backstory="You are an expert at finding and analyzing relevant information from large document collections. You excel at identifying key passages and extracting actionable insights.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[WebsiteSearchTool()]
)

# Analysis Agent
analysis_agent = Agent(
    role="Domain Expert",
    goal="Perform deep analysis on retrieved documents using domain expertise",
    backstory="You are a domain expert with deep knowledge in business, technology, and strategy. You excel at identifying patterns, drawing connections, and providing expert insights.",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# Synthesis Agent
synthesis_agent = Agent(
    role="Strategic Synthesizer",
    goal="Combine multiple analysis results into comprehensive, actionable recommendations",
    backstory="You are a strategic thinker who excels at combining diverse inputs into clear, actionable recommendations. You have a talent for identifying the most important insights and presenting them clearly.",
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [ ]:
# Research Task
research_task = Task(
    description="Research the given topic by retrieving relevant documents and extracting key information. Focus on finding authoritative sources and identifying the most relevant passages.",
    agent=research_agent,
    expected_output="A structured summary of relevant documents with key quotes and source references."
)

# Analysis Task
analysis_task = Task(
    description="Analyze the research findings using domain expertise. Identify patterns, implications, and potential opportunities or risks.",
    agent=analysis_agent,
    expected_output="A detailed analysis with expert insights, implications, and recommendations."
)

# Synthesis Task
synthesis_task = Task(
    description="Synthesize the research and analysis into a comprehensive response that addresses the original query with actionable recommendations.",
    agent=synthesis_agent,
    expected_output="A comprehensive response with clear recommendations and supporting evidence."
)

In [ ]:
# Create the crew
analysis_crew = Crew(
    agents=[research_agent, analysis_agent, synthesis_agent],
    tasks=[research_task, analysis_task, synthesis_task],
    verbose=2,
    process="sequential"  # or "hierarchical" for complex scenarios
)

# Execute the workflow
def run_analysis(query):
    result = analysis_crew.kickoff(inputs={"topic": query})
    return result